# 08 · Cost Analysis

> **Module 10 · Pricing & Cost Optimization · Lectures 10.1–10.2**

In this notebook you will:

- Understand the four billable dimensions
- Measure what your queries actually cost
- Build a cost model for a real workload
- Apply each optimisation and measure the saving
- Set up a cost guardrail that blocks expensive queries

---

## Setup

In [ ]:
import os
import sys

sys.path.insert(0, ".")
from bq_client import get_client, run_query, run_to_dataframe, estimate_cost

from google.cloud import bigquery

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "YOUR_PROJECT_ID")
client = get_client(project=PROJECT_ID)
print(f"Project: {client.project}")

## 1. The four billable dimensions

| Dimension | Charged for | Typical rate |
|-----------|-------------|--------------|
| **Storage** | Logical bytes | $0.02 / GiB / month (active) |
| **Query processing** | Bytes scanned | $6.25 / TiB (on-demand) |
| **DML** | Bytes scanned **and** bytes modified | $6.25 / TiB + $0.01 / GiB |
| **DDL / APIs** | Schema updates, table ops | $0.01 / GiB modified |

In [ ]:
PRICE_PER_TIB = 6.25
STORAGE_PER_GIB = 0.02
DML_PER_GIB = 0.01

FREE_QUERY_TIB_MONTH = 1
FREE_STORAGE_GB_MONTH = 10


def fmt_cost(usd: float) -> str:
    return f"${usd:,.6f}"


print("Cost model constants loaded")
print(f"  Query:            ${PRICE_PER_TIB}/TiB scanned")
print(f"  Storage (active): ${STORAGE_PER_GIB}/GiB/month")
print(f"  DML modify:       ${DML_PER_GIB}/GiB")
print(f"  Free tier:        {FREE_QUERY_TIB_MONTH} TiB queries/month, {FREE_STORAGE_GB_MONTH} GB storage")

## 2. Measure what you are already spending

`INFORMATION_SCHEMA.JOBS_BY_PROJECT` tells you exactly where the bytes went.

In [ ]:
df = run_to_dataframe(
    f"""
    SELECT
      user_email,
      COUNT(*) AS query_count,
      ROUND(SUM(total_bytes_billed) / POWER(1024, 4) * 6.25, 4) AS cost_usd,
      ROUND(AVG(total_bytes_billed) / POWER(1024, 4) * 6.25, 6) AS avg_cost_usd,
      ROUND(MAX(total_bytes_billed) / POWER(1024, 4) * 6.25, 4) AS max_cost_usd
    FROM `{PROJECT_ID}.`region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
    WHERE job_type = 'QUERY'
      AND state = 'DONE'
      AND creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
    GROUP BY user_email
    ORDER BY cost_usd DESC
    LIMIT 20
    """
)
display(df)

In [ ]:
# The most expensive queries — the ones to fix first
df = run_to_dataframe(
    f"""
    SELECT
      job_id,
      user_email,
      ROUND(total_bytes_billed / POWER(1024, 4) * 6.25, 4) AS cost_usd,
      ROUND(total_bytes_billed / POWER(1024, 3), 1) AS billed_mb,
      creation_time
    FROM `{PROJECT_ID}.`region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
    WHERE job_type = 'QUERY'
      AND total_bytes_billed > 1073741824
      AND creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
    ORDER BY total_bytes_billed DESC
    LIMIT 20
    """
)
display(df)

In [ ]:
# The cache hit rate — a free win
df = run_to_dataframe(
    f"""
    SELECT
      ROUND(100 * COUNTIF(cache_hit) / COUNT(*), 2) AS cache_hit_pct,
      COUNT(*) AS total_queries,
      ROUND(SUM(IF(cache_hit, 0, total_bytes_billed)) / POWER(1024, 4) * 6.25, 4) AS paid_cost_usd
    FROM `{PROJECT_ID}.`region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
    WHERE job_type = 'QUERY'
      AND creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
    """
)
display(df)

In [ ]:
# Cost by label — cost attribution
df = run_to_dataframe(
    f"""
    SELECT
      labels.tier AS tier,
      labels.env AS env,
      COUNT(*) AS query_count,
      ROUND(SUM(total_bytes_billed) / POWER(1024, 4) * 6.25, 4) AS cost_usd
    FROM `{PROJECT_ID}.`region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
    WHERE job_type = 'QUERY'
      AND creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
    GROUP BY tier, env
    ORDER BY cost_usd DESC
    """
)
display(df)

## 3. Storage cost by dataset

Storage is usually the smaller number, but it grows silently. Measure it.

In [ ]:
df = run_to_dataframe(
    f"""
    SELECT
      table_schema AS dataset,
      COUNT(*) AS table_count,
      ROUND(SUM(size_bytes) / POWER(1024, 3), 2) AS size_gb,
      ROUND(SUM(size_bytes) / POWER(1024, 3) * 0.02, 2) AS est_monthly_storage_usd,
      ROUND(SUM(row_count) / 1000000, 2) AS total_rows_m
    FROM `{PROJECT_ID}.`region-us`.INFORMATION_SCHEMA.TABLE_STORAGE
    GROUP BY dataset
    ORDER BY size_gb DESC
    """
)
display(df)

## 4. Build a cost model for a workload

A model with low/base/high scenarios, so you can plan against a budget.

In [ ]:


def model_workload(
    *,
    raw_gb=800,
    curated_gb=900,
    mart_gb=120,
    daily_scan_gb=120,
    daily_dml_modified_gb=60,
    events_per_day=400_000_000,
    bytes_per_event=500,
    bi_scan_gb_per_day=200,
    days=30,
):
    """Model the monthly BigQuery cost of a pipeline."""
    # Storage
    storage = (
        raw_gb * STORAGE_PER_GIB
        + curated_gb * STORAGE_PER_GIB * 0.5     # long-term pricing
        + mart_gb * STORAGE_PER_GIB
    )

    # Batch queries (daily_scan_gb is GB, convert to TiB)
    daily_query = (daily_scan_gb / 1024) * PRICE_PER_TIB
    daily_dml = (daily_scan_gb / 1024) * PRICE_PER_TIB + daily_dml_modified_gb * DML_PER_GIB
    monthly_query = (daily_query + daily_dml) * days

    # Streaming (first 1 GB/day/table is free)
    stream_gb = events_per_day * bytes_per_event / 1024**3
    billable_gb = max(0, stream_gb - 1)
    monthly_stream = billable_gb * days * (DML_PER_GIB / 0.2)   # $0.01 per 200 MB

    # BI
    monthly_bi = (bi_scan_gb_per_day / 1024) * PRICE_PER_TIB * days

    return {
        "storage": storage,
        "queries": monthly_query,
        "streaming": monthly_stream,
        "bi": monthly_bi,
        "total": storage + monthly_query + monthly_stream + monthly_bi,
    }


SCENARIOS = {
    "low":  dict(events_per_day=150_000_000, daily_scan_gb=60,  bi_scan_gb_per_day=80),
    "base": dict(events_per_day=400_000_000, daily_scan_gb=120, bi_scan_gb_per_day=200),
    "high": dict(events_per_day=900_000_000, daily_scan_gb=300, bi_scan_gb_per_day=500),
}

results = {name: model_workload(**kwargs) for name, kwargs in SCENARIOS.items()}

print(f"{'component':<22} {'low':>14} {'base':>14} {'high':>14}")
print("-" * 68)
for key in ("storage", "queries", "streaming", "bi", "total"):
    print(f"{key:<22} " + " ".join(f"{fmt_cost(results[s][key]):>14}" for s in ("low", "base", "high")))

print(f"\nMonthly budget check (budget $1,200):")
for name, r in results.items():
    status = "OK" if r["total"] <= 1200 else "OVER BUDGET"
    print(f"  {name:<6} {fmt_cost(r['total']):>12}  {status}")

## 5. On-demand vs capacity pricing

Capacity is a **commitment**. Find the crossover point before you switch.

In [ ]:
def capacity_vs_on_demand(monthly_on_demand: float, slots: int = 100,
                           rate: float = 0.04, hours_per_month: int = 720):
    """Compare capacity pricing with on-demand at a given slot count."""
    slot_hours = slots * hours_per_month
    capacity_cost = slot_hours * rate
    breakeven_slot_hours = monthly_on_demand / rate
    breakeven_slots = breakeven_slot_hours / hours_per_month
    return {
        "slot_hours": slot_hours,
        "capacity_cost": capacity_cost,
        "on_demand_cost": monthly_on_demand,
        "cheaper": "capacity" if capacity_cost < monthly_on_demand else "on-demand",
        "breakeven_slots": breakeven_slots,
    }


print(f"{'monthly on-demand':>20} {'slots':>8} {'capacity cost':>16} {'cheaper':>12} {'breakeven slots':>18}")
print("-" * 82)
for on_demand in (500, 1000, 2000, 4200, 10000):
    for slots in (100, 300, 1000):
        r = capacity_vs_on_demand(on_demand, slots)
        print(f"{fmt_cost(on_demand):>20} {slots:>8} {fmt_cost(r['capacity_cost']):>16} "
              f"{r['cheaper']:>12} {r['breakeven_slots']:>18,.0f}")
    print()

## 6. Apply each optimisation and measure

Every optimisation below is applied to a real query, with the byte count measured
before and after.

In [ ]:
T = f"`{PROJECT_ID}.curated.orders_clustered`"

OPTIMISATIONS = {
    "0. baseline: SELECT *":        f"SELECT * FROM {T}",
    "1. + partition filter":          f"SELECT * FROM {T} WHERE order_date = DATE '2026-01-15'",
    "2. + specific columns":         f"SELECT order_id, customer_id, total_amount FROM {T} WHERE order_date = DATE '2026-01-15'",
    "3. + cluster filter":            f"SELECT order_id, total_amount FROM {T} WHERE order_date = DATE '2026-01-15' AND customer_id = 'c0007'",
    "4. partition decorator":         f"SELECT order_id, total_amount FROM `{PROJECT_ID}.curated.orders_clustered$20260115` WHERE customer_id = 'c0007'",
}

print(f"{'optimisation':<32} {'bytes':>14} {'saved vs baseline':>20}")
print("-" * 68)
baseline = None
results = {}
for label, sql in OPTIMISATIONS.items():
    est = estimate_cost(sql)
    results[label] = est["bytes_processed"]
    if baseline is None:
        baseline = est["bytes_processed"]
    saving = 1 - (est["bytes_processed"] / baseline) if baseline else 0
    print(f"{label:<32} {est['bytes_processed']:>14,} {saving:>19.1%}")

In [ ]:
# The view vs the MV, measured
VIEW_SQL = f"""
SELECT order_date, region, channel, COUNT(*) AS n
FROM `{PROJECT_ID}.curated.v_active_orders`
WHERE order_date = DATE '2026-01-15'
GROUP BY order_date, region, channel
"""

MV_SQL = f"""
SELECT order_date, region, channel, order_count AS n
FROM `{PROJECT_ID}.mart.mv_daily_revenue`
WHERE order_date = DATE '2026-01-15'
"""

view_est = estimate_cost(VIEW_SQL)
mv_est = estimate_cost(MV_SQL)

print(f"{'source':<12} {'bytes':>14} {'est. cost':>14}")
print("-" * 42)
print(f"{'view':<12} {view_est['bytes_processed']:>14,} {fmt_cost(view_est['estimated_cost_usd']):>14}")
print(f"{'MV':<12} {mv_est['bytes_processed']:>14,} {fmt_cost(mv_est['estimated_cost_usd']):>14}")

if view_est["bytes_processed"]:
    print(f"\nMV saving: {1 - mv_est['bytes_processed'] / view_est['bytes_processed']:.1%} fewer bytes")

## 7. Storage optimisations

Long-term storage halves the price of cold data. Capacity-optimized tables cut
scan costs by ~30% at roughly double the storage.

In [ ]:
STORAGE_MODELS = {
    "standard (active)":        {"per_gib": 0.02, "scan_multiplier": 1.0},
    "standard (long-term)":    {"per_gib": 0.01, "scan_multiplier": 1.0},
    "capacity-optimized":      {"per_gib": 0.04, "scan_multiplier": 0.7},
}

SIZE_GB = 5_000
QUERIES_PER_MONTH = 500
TB_SCANNED_PER_QUERY = 0.5

print(f"{'model':<24} {'storage/mo':>12} {'query/mo':>12} {'total/mo':>12}")
print("-" * 62)
for name, m in STORAGE_MODELS.items():
    storage = SIZE_GB * m["per_gib"]
    query = QUERIES_PER_MONTH * TB_SCANNED_PER_QUERY * PRICE_PER_TIB * m["scan_multiplier"]
    print(f"{name:<24} {fmt_cost(storage):>12} {fmt_cost(query):>12} {fmt_cost(storage + query):>12}")

print("\nCapacity-optimized wins when a table is stored once and scanned rarely.")

In [ ]:
# Apply storage optimisations to a real table
table = client.get_table(f"{PROJECT_ID}.curated.orders_part")
print(f"Current storage model: {table.storage_billing_model}")

# Move to long-term storage
run_query(
    f"ALTER TABLE `{PROJECT_ID}.curated.orders_part` "
    "SET OPTIONS (storage_billing_model = 'STORAGE_BILLING_MODEL_LONG_TERM')"
)
print("✓ Moved to long-term storage")

# Back to standard
run_query(
    f"ALTER TABLE `{PROJECT_ID}.curated.orders_part` "
    "SET OPTIONS (storage_billing_model = 'STORAGE_BILLING_MODEL_STANDARD')"
)
print("✓ Back to standard storage")

## 8. Build a cost guardrail

A decorator that dry-runs a query and raises if it would cost too much. Apply it to
every query function in a pipeline.

In [ ]:
import functools


class QueryTooExpensive(Exception):
    """Raised when a query would exceed the configured cost threshold."""


def cost_guardrail(max_usd: float = 1.00, label: str = ""):
    """Decorator: dry-run the query and raise if it exceeds the budget."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(sql, *args, **kwargs):
            est = estimate_cost(sql)
            cost = est["estimated_cost_usd"]
            bytes_ = est["bytes_processed"]
            if cost > max_usd:
                raise QueryTooExpensive(
                    f"Query would cost ${cost:.4f} (limit ${max_usd:.2f}). "
                    f"Bytes: {bytes_:,}. Add a partition filter. "
                    f"Label: {label or func.__name__}"
                )
            return func(sql, *args, **kwargs)
        return wrapper
    return decorator


@cost_guardrail(max_usd=0.01, label="cheap query")
def run_guarded(sql):
    return run_query(sql)


@cost_guardrail(max_usd=0.01, label="expensive query")
def run_expensive(sql):
    return run_query(sql)


# The cheap query passes
try:
    rows = run_guarded(f"SELECT COUNT(*) FROM {T} WHERE order_date = DATE '2026-01-15'")
    print(f"✅ Cheap query allowed: {rows[0][0]} rows")
except QueryTooExpensive as exc:
    print(f"❌ {exc}")

# The expensive query is blocked
try:
    run_expensive(f"SELECT * FROM {T}")
    print("⚠️  Query was NOT blocked")
except QueryTooExpensive as exc:
    print(f"✅ Blocked: {exc}")

## 9. The optimisation checklist

| # | Lever | Typical saving | Effort |
|---|-------|---------------|--------|
| 1 | Partition filters | 90–99% | Low |
| 2 | Avoid `SELECT *` | 50–90% | Low |
| 3 | Materialized views | 80–95% | Medium |
| 4 | Clustering | 30–70% | Medium |
| 5 | BI Engine | 90%+ (reserved) | Medium |
| 6 | Storage expiration / long-term | 50% of storage | Low |
| 7 | Capacity-optimized tables | ~30% (on scans) | Low |
| 8 | Query cache | 100% for repeats | Low |

## 🔑 Takeaways

- Four billable dimensions: **storage, query processing, DML, DDL/APIs**.
- **Dry run everything.** `total_bytes_processed` is exact and free.
- **`maximum_bytes_billed` on every job** — a failed query is free.
- `INFORMATION_SCHEMA.JOBS_BY_PROJECT` is your ground truth for where the money went.
- **Partition filters first**, then column pruning, then clustering, then MVs.
- `require_partition_filter = TRUE` prevents the expensive mistake.
- **Capacity pricing only above ~500–1,000 slot-hours/month** of stable usage.
- Build a **cost guardrail decorator** so an expensive query fails in CI, not on the
  invoice.

---

## 📚 Next

[09 · External Tables and File Formats](./09-external-tables-and-file-formats.ipynb)